In [1]:
from dotenv import load_dotenv
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [2]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

In [8]:
@tool
def get_product(name: str) -> str:
    """Look up a product by name and return its price, rating, stock, and description. Return of description is important. It will definitely return some description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)

In [12]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent = create_agent(
    llm,
    tools=[get_product],
    system_prompt="You are a helpful product assistant for an online tech store. When looking up a product, always include its full description, price, and rating in your response.",
)

In [13]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [14]:
ask("what is the price of wireless headphones.")

The price of the wireless headphones is $79.99. They have a rating of 4.6 and come with over-ear Bluetooth, 30-hour battery life, and active noise cancellation.


In [15]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)

In [16]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
review_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

review_agent = create_agent(
    review_llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

def ask2(question: str):
    result = review_agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [17]:
ask2("how do people like smart watch")

People like smart watches because they are convenient and provide a range of useful features such as tracking heart rate and sleep, and being water-resistant. The smart watch has a rating of 4.3 and is priced at $199.99. It also has a long battery life, lasting up to 5 days. Overall, people have given the smart watch a positive review, with an average rating of 3.9 out of 5 stars based on 340 reviews.


In [18]:
from langgraph.checkpoint.memory import InMemorySaver

In [19]:
llm_for_memory = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
agent_with_memory = create_agent(
    review_llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
    checkpointer=InMemorySaver()
)

def ask_with_memory(question: str):
    config = {"configurable": {"thread_id": "user-alice-session-1"}}
    result = agent_with_memory.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config
    )
    print(result["messages"][-1].content)


In [ ]:
ask_with_memory("W")